**Question:** Can a pretrained tabular foundation model (TabPFN v2, in-context, no training) match or beat the production GRU (C3) and the ELO baseline?

**Assumptions:** Same test matches as ELO/C3 (fold 4 of the walk-forward split); context limited to 10,000 training rows; all predictions isotonic-calibrated on the calibration slice.

**Method:** Three feature variants x two context selections x three seeds, paired bootstrap by match against ELO and C3, plus walk-forward folds 1-4.

**Result:** TabPFN (D-B') is significantly better than ELO (dBrier about -0.014) but not better than C3 (about +0.002 worse), so it does not go to production.

# 14 -- Exp D: TabPFN v2 (in-context tabular foundation model) vs ELO and the enriched GRU (C3)

**Idea.** TabPFN v2 is a transformer pretrained on synthetic tabular tasks. It does no gradient
training on our data: the training rows are passed as *context* and the model predicts the test
rows in a single forward pass (in-context learning).

**Setup (all runs).**
- Hardware: Colab T4 GPU, `n_estimators=8`.
- Context: at most **10,000 rows**, either the most recent 10k training rows (`last10k`) or a random 10k (`random10k`).
- Calibration: isotonic regression fitted on the calibration slice (all numbers below are calibrated probabilities).
- Variants: **D-A'** (A' features, no ELO), **D-B'** (B' features, with `elo_diff`), **D-B'-par** (B' minus the match-context columns that do not exist for an arbitrary player pair, i.e. a production-compatible pair model).
- Seeds: 42, 1, 2. We report the per-seed mean/min/max and the 3-seed ensemble (mean of predictions).
- Main split = walk-forward fold 4 (same matches as ELO and C3). Walk-forward folds 1-3 are run for `last10k`, seed 42 only.

**References** use exactly the same match rows: ELO = `1/(1+10^(-elo_diff/400))`, C3 = seed 42 of the
enriched GRU, C3ens = mean of 5 C3 seeds.

**Statistics.** Paired bootstrap by match (2,000 resamples, seed 42, both perspective rows of a match
travel together); CORP decomposition (Brier = MCB - DSC + UNC; Dimitriadis, Gneiting, Jordan 2021).

**Licence note.** TabPFN v2 is used for this experiment (its weights permit commercial use, so it is the only
version eligible for production). The newest release, TabPFN-3.5 (non-commercial licence), is evaluated separately
as a research-only comparison in notebook 15; it is never used in production.

The data directory is the same working directory as notebooks 01-13 (`../data`); the TabPFN run
outputs sit next to it in `../exp_d`. No data is stored in this notebook.

## Step 0 -- Imports and data

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, '../scripts')
from exp_d_analysis import (load_folds, met, main_split_table, paired_dbrier, corp_delta, rest_table,
                            boot_idx, VARIANTS, CTXS)

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', None)

WORKDIR = '..'
B = load_folds(WORKDIR)
f4 = B[4].copy()
y = f4.Label.values
print('folds loaded:', sorted(B), '| fold 4 matches:', f4.SusretId.nunique(), '| rows:', len(f4))

## Step 1 -- Main split (fold 4): 3 seeds + ensemble vs ELO / C3 / C3ens

In [ ]:
R = main_split_table(f4)
cols = ['variant', 'ctx', 'brier_mean', 'brier_min', 'brier_max', 'ens_brier', 'ens_auc', 'ens_acc', 'ens_logloss', 'ens_ece']
print(R[cols].round(5).to_string(index=False))

refs = pd.DataFrame({n: met(f4['p_' + n].values, y) for n in ['ELO', 'C3', 'C3ens']}).T
print()
print(refs[['brier', 'auc', 'acc', 'logloss', 'ece']].round(5))

**Reading the table.** D-B' and D-B'-par (which see `elo_diff`) land at Brier ~0.169, well below ELO
(0.1825) but above C3 (0.1669) and C3ens (0.1651). D-A' (no ELO) is indistinguishable from ELO.
`last10k` is better than `random10k`; the seed-to-seed spread is small (about 0.0005).

## Step 2 -- Paired bootstrap by match (delta Brier, D ensemble minus reference)

In [ ]:
boots4 = boot_idx(f4.SusretId.values)
out = []
for v in VARIANTS:
    for c in CTXS:
        pe = f4[f'p_{v}__{c}__ens'].values
        for ref in ['ELO', 'C3', 'C3ens']:
            mu, lo, hi = paired_dbrier(pe, f4['p_' + ref].values, y, boots4)
            out.append((v, c, ref, mu, lo, hi))
CI = pd.DataFrame(out, columns=['variant', 'ctx', 'vs', 'dBrier', 'lo95', 'hi95'])
print(CI.round(5).to_string(index=False))

Negative = D better. Against ELO the D-B' variants are clearly better (CI entirely below 0). Against
C3 (seed 42) the CI includes 0 for `last10k`; against the 5-seed C3 ensemble D is worse (CI above 0).

In [ ]:
best = R.sort_values('ens_brier').iloc[0]
bv = f'{best.variant}__{best.ctx}'
pb = f4[f'p_{bv}__ens'].values
print('best config:', bv)
for ref in ['ELO', 'C3']:
    d = corp_delta(pb, f4['p_' + ref].values, y, boots4)
    print(f'CORP vs {ref}:', {k: tuple(round(x, 5) for x in v) for k, v in d.items()}, '(mean, lo95, hi95)')

CORP: versus ELO, D lowers miscalibration (dMCB < 0) and raises discrimination (dDSC > 0). Versus C3
neither difference is significant; D has slightly lower discrimination (dDSC < 0) and the CIs include 0.

## Step 3 -- By gender and by rest length (fold 4, best config)

In [ ]:
for name, g in (('SM (men)', 0), ('SF (women)', 1)):
    mk = (f4.liga_tip == g).values
    bt = boot_idx(f4.SusretId.values[mk]); yy = y[mk]
    brier = {n: np.mean((p[mk] - yy) ** 2) for n, p in (('ELO', f4.p_ELO.values), ('C3', f4.p_C3.values), ('D', pb))}
    dC3 = paired_dbrier(pb[mk], f4.p_C3.values[mk], yy, bt)
    dEL = paired_dbrier(pb[mk], f4.p_ELO.values[mk], yy, bt)
    print(name, 'matches', mk.sum() // 2, {k: round(v, 4) for k, v in brier.items()})
    print('   D-C3  ', tuple(round(x, 5) for x in dC3))
    print('   D-ELO ', tuple(round(x, 5) for x in dEL))

In [ ]:
print(rest_table(f4, f'p_{bv}__ens').round(3).to_string(index=False))

For long breaks (>180 days) D is closer to the observed win rate than C3, but those buckets contain
only 50-80 rows each, too few to conclude anything.

## Step 4 -- Walk-forward folds 1-4 (`last10k`, seed 42)

In [ ]:
wfv = [f'{v}__last10k__s42' for v in VARIANTS]
rows = []
for k in (1, 2, 3, 4):
    d = B[k]; yy = d.Label.values
    r = {'fold': k, 'n_matches': d.SusretId.nunique()}
    for n in ['ELO', 'C3', 'C3ens'] + wfv:
        r[n.replace('__last10k__s42', '')] = np.mean((d['p_' + n] - yy) ** 2)
    rows.append(r)
T = pd.DataFrame(rows)
print(T.round(5).to_string(index=False))
print()
print(T.drop(columns=['fold', 'n_matches']).mean().round(5).rename('mean Brier'))
for v in ['D-Bprime', 'D-Bprime-par']:
    for ref in ['ELO', 'C3']:
        print(v, 'better than', ref, 'in', int((T[v] < T[ref]).sum()), '/ 4 folds')

In [ ]:
pool = pd.concat([B[k].assign(SusretId=B[k].SusretId.map(lambda s, k=k: f'{k}_{s}')) for k in B], ignore_index=True)
yp = pool.Label.values
bp = boot_idx(pool.SusretId.values)
print('pooled matches:', pool.SusretId.nunique())
for v in wfv[1:]:
    for ref in ['ELO', 'C3']:
        mu, lo, hi = paired_dbrier(pool['p_' + v].values, pool['p_' + ref].values, yp, bp)
        print(f'POOLED {v} vs {ref}: dBrier {mu:+.5f} [{lo:+.5f}, {hi:+.5f}]')

## Step 5 -- Brier by fold: ELO vs C3 vs D-B'

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
x = np.arange(len(T)); w = 0.27
ax.bar(x - w, T['ELO'], w, label='ELO', color='#374151')
ax.bar(x, T['C3'], w, label='C3 (GRU)', color='#ea580c')
ax.bar(x + w, T['D-Bprime'], w, label="D-B' (TabPFN v2)", color='#2563eb')
ax.set_xticks(x); ax.set_xticklabels([f'fold {k}' for k in T['fold']])
ax.set_ylim(0.13, 0.19); ax.set_ylabel('Brier score (lower is better)')
ax.set_title('Walk-forward Brier by fold')
ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.show()

## Conclusion

- **TabPFN v2 (D-B' / D-B'-par, `last10k`) is significantly better than ELO**: dBrier about -0.014 on the main split (95% CI entirely below 0) and better in 4/4 walk-forward folds.
- **It is not better than the production GRU (C3).** On the main split the difference is about +0.002 (CI includes 0); in walk-forward C3 wins 3/4 folds and the pooled difference is about +0.002 to +0.0025 (CI above 0); against the 5-seed C3 ensemble D is worse on the main split as well.
- D-A' (no ELO features) is no better than ELO; `last10k` context beats `random10k`.
- Decision: the production criterion (D-B'-par beats C3, in at least 3/4 folds, CI below 0) is **not met**, so Exp D does **not** go to production.

**Limitations.**
- TabPFN v2 is run with a 10,000-row context; on fold 4 that is only about 41% of the available training rows, so the comparison with a GRU trained on all rows is not fully like-for-like. More context was not tested.
- TabPFN v2 is used for this experiment (its weights permit commercial use, so it is the only version eligible for production). The newest release, TabPFN-3.5 (non-commercial licence), is evaluated separately as a research-only comparison in notebook 15; it is never used in production.
- Walk-forward folds 1-3 use one seed and one context selection only.
- Raw vs isotonic-calibrated TabPFN output was not compared separately; all numbers are calibrated.